In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/training"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/dataset-test")

In [ ]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")


## Graph Augment


In [ ]:
ID = 0
m, digraph, _ = dataset.jppype_show(ID, augment=False)
m

In [ ]:
dataset.get(20, augment=True, version="fvt")
%timeit dataset.get(20, augment=True, version="fvt")

In [ ]:
subset = dataset.split(list(range(20))).preload()
%timeit [subset.get(i, augment=True, version="fvt") for i in range(3)]

In [ ]:
from fundus_vessels_toolkit.utils.profiling import Profiler, watch

with Profiler():
    for i in range(len(subset)):
        subset.get(i, augment=True, version="fvt")

In [ ]:
subset.get(0, augment=True, version="fvt")
%timeit subset.get(0, augment=True, version='fvt')

In [ ]:
import cProfile

dataset.preload()

In [ ]:
cProfile.run("dataset.get(20, augment=True, version='fvt')", sort="cumulative")

In [ ]:
from fundus_vessels_toolkit.utils.profiling import ProfilerWatch

ProfilerWatch.reset()
dataset.get(20, augment=True, version="fvt")
print(ProfilerWatch.get("split_branch").print())

In [ ]:
from fundus_toolkits.transform import ElasticTransform, ElasticTransformLegacy

data = dataset.get(20, augment=False, version="fvt")

elastic = ElasticTransform.random(data.img.shape[-2:], 80, 200)
elastic_leg = ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)


%timeit ElasticTransform.random(data.img.shape[-2:], 80, 200)
%timeit ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)

In [ ]:
sample = dataset.get_sample(20)

In [ ]:
%timeit sample.graphes['fvt'].transform(elastic)
%timeit sample.graphes['fvt'].transform(elastic_leg)

In [ ]:
%timeit elastic.warp(data.img.permute(1, 2, 0))
%timeit elastic_leg.warp(data.img.permute(1, 2, 0).numpy())

In [ ]:
from fundus_vessels_toolkit.utils.profiling import watch, ProfilerWatch

elastic = ElasticTransform.random(data.img.shape[-2:], 80, 200)
elastic_leg = ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)
ProfilerWatch.reset()
elastic.warp(data.img.permute(1, 2, 0))
elastic_leg.warp(data.img.permute(1, 2, 0).numpy())
sample.graphes["fvt"].transform(elastic)
sample.graphes["fvt"].transform(elastic_leg)
print(watch("warp").print())
print(watch("ElasticTransformLegacy._warp").print())
print(watch("ElasticTransform._transform inverse").print())
print(watch("ElasticTransformLegacy._transform inverse").print())